In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import time
import math
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
import random
import os

In [ ]:
from pathlib import Path

dataset_name = "youtube_pedestrian.csv"
data_file = Path.cwd().parents[1] / "data" / "filtered" / dataset_name

df = pd.read_csv(data_file, parse_dates=["DATE"])
print("Loaded:", data_file)

In [ ]:
resolution_horizon_map = {

    "2000ms": 5,
    "3000ms": 4,    
}


def resample_data(df_original, freq):
    df_resampled = df_original.copy()

    df_resampled["DATE"] = pd.to_datetime(df_resampled["DATE"])

    df_resampled = df_resampled.set_index("DATE")
    df_resampled = df_resampled.resample(freq).mean()
    df_resampled = df_resampled.round().astype(int).reset_index()

    return df_resampled

In [ ]:
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

In [ ]:
def prepare_chronos_dataframe(df):
    df = df.copy()

    df = df.rename(
        columns={
            "ue_ident": "item_id",
            "DATE": "timestamp",
            "mac_dl_brate": "target",
        }
    )

    return df

In [ ]:
def rolling_chronos_forecast_all(
    predictor,
    test_data,
    prediction_length,
    stride=1,
    measure_time=False,
):
    results = []
    times = []

    if not isinstance(test_data.index, pd.MultiIndex):
        raise ValueError("Expected test_data with MultiIndex: item_id, timestamp.")

    freq = test_data.index.get_level_values("timestamp").to_series().diff().mode()[0]
    print("Inferred frequency:", freq)

    for item_id, series in test_data.groupby(level="item_id"):
        series = series.reset_index()

        for start in range(0, len(series) - prediction_length, stride):
            context = series.iloc[: start + prediction_length]
            context_df = context.set_index(["item_id", "timestamp"])

            last_ts = context["timestamp"].iloc[-1]

            future_ts = pd.date_range(
                start=last_ts + pd.Timedelta(freq),
                periods=prediction_length,
                freq=freq,
            )

            future_cov_df = pd.DataFrame({
                "item_id": item_id,
                "timestamp": future_ts,
            })

            for col in predictor.known_covariates_names:
                last_val = context[col].iloc[-1]
                future_cov_df[col] = last_val

            future_cov_tdf = TimeSeriesDataFrame(
                future_cov_df.set_index(["item_id", "timestamp"])
            )

            t0 = time.time()
            forecast = predictor.predict(
                context_df,
                known_covariates=future_cov_tdf,
            )
            t1 = time.time()

            if measure_time:
                times.append(t1 - t0)

            forecast_mean = forecast.loc[item_id]["mean"].to_numpy().flatten()

            df_forecast = pd.DataFrame({
                "item_id": item_id,
                "timestamp": future_ts,
                "mean": forecast_mean,
            })

            results.append(df_forecast)

    forecasts_df = pd.concat(results, ignore_index=True)

    return (forecasts_df, times) if measure_time else forecasts_df

In [ ]:
def run_chronos_for_resolution(df_original, resolution, prediction_length):
    print("=" * 70)
    print("Resolution:", resolution)
    print(f"Running Chronos for resolution={resolution}, horizon={prediction_length}")

    df_resampled = resample_data(df_original, resolution)

    print("Resampled shape:", df_resampled.shape)

    df_chronos = prepare_chronos_dataframe(df_resampled)

    uni_data = TimeSeriesDataFrame(df_chronos)

    train_size = int(len(df_chronos) * 0.8)

    train_data = uni_data.iloc[:train_size]
    test_data = uni_data.iloc[train_size:]

    predictor = TimeSeriesPredictor(
        prediction_length=prediction_length,
        target="target",
        known_covariates_names=[
            "mac_dl_cqi",
            "mac_dl_mcs",
            "mac_dl_ok",
            "mac_dl_nok",
        ],
        eval_metric="MAE",
        freq=resolution,
    ).fit(
        train_data,
        hyperparameters={
            "Chronos": [
                {
                    "model_path": "bolt_small",
                    "ag_args": {"name_suffix": "ZeroShot"},
                }
            ]
        },
        enable_ensemble=False,
    )

    rolling_preds = rolling_chronos_forecast_all(
        predictor=predictor,
        test_data=test_data,
        prediction_length=prediction_length,
        stride=1,
        measure_time=False,
    )

    actual = test_data.reset_index()[["item_id", "timestamp", "target"]]

    aligned = rolling_preds.merge(
        actual,
        on=["item_id", "timestamp"],
        how="inner",
    )

    if aligned.empty:
        print(f"No aligned predictions for {resolution}. Skipping.")
        return None

    scaler = MinMaxScaler()
    scaler.fit(train_data.reset_index()[["target"]])

    aligned["mean_scaled"] = scaler.transform(
        aligned[["mean"]].to_numpy()
    )

    aligned["target_scaled"] = scaler.transform(
        aligned[["target"]].to_numpy()
    )

    rmse_scaled = np.sqrt(
        mean_squared_error(
            aligned["target_scaled"],
            aligned["mean_scaled"],
        )
    )

    mae_scaled = mean_absolute_error(
        aligned["target_scaled"],
        aligned["mean_scaled"],
    )

    print(f"Scaled RMSE: {rmse_scaled:.3f}")
    print(f"Scaled MAE : {mae_scaled:.3f}")

    return {
        "model": "Chronos",
        "variant": "zero_shot",
        "dataset": "youtube_pedestrian",
        "resolution": resolution,
        "prediction_horizon": prediction_length,
        "scaled_rmse": rmse_scaled,
        "scaled_mae": mae_scaled,
    }

In [ ]:
all_chronos_results = []
df_base = df.copy()

for resolution, prediction_length in resolution_horizon_map.items():
    result = run_chronos_for_resolution(
        df_original=df_base,
        resolution=resolution,
        prediction_length=prediction_length,
    )

    if result is not None:
        all_chronos_results.append(result)

chronos_temporal_results = pd.DataFrame(all_chronos_results)

In [ ]:
chronos_temporal_results.head()

In [ ]:
results_dir = Path.cwd().parent / "results" / "metrics" / "temporal_resolution"
results_dir.mkdir(parents=True, exist_ok=True)

metrics_file = results_dir / "chronos.csv"

chronos_temporal_results.to_csv(metrics_file, index=False)

print("Saved metrics to:", metrics_file)